# Modelo SIR com agentes

## Importando bibliotecas e definindo funções

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm import tqdm
import agent_based_model as abm
from scipy.optimize import curve_fit
import functools as ft

In [ ]:
def rmsd_func(a1, a2):
    rmsd = np.sqrt(np.mean((a1 - a2)**2))
    return rmsd

def identify_ext(infected_serie):
    if infected_serie[-1] == 0:
        return 1
    else:
        return 0

def rmsd_func_advance(y1, y2, x1 = False, x2 = False, mode = 'simple'):

    if np.all(np.isnan(y1)) or np.all(np.isnan(y2)):

        print('All NaN')
        return np.nan, np.nan, np.nan

    x1_interval = x1[-1] - x1[0]
    x2_interval = x2[-1] - x2[0]

    # The serie with smaller interval will dictate the time instante to evaluate RMSD
    if x1_interval > x2_interval:
        broader_x = x1
        narrower_x = x2
        y_to_cut = y1
        y_to_maintain = y2
    else:
        broader_x = x2
        narrower_x = x1
        y_to_cut = y2
        y_to_maintain = y1

    
    match mode:

        case 'simple':
            
            return rmsd_func(y1, y2)

        case 'start_at_matching_x':

            match_xdelta_condition = np.isin(broader_x.round(3), narrower_x.round(3))
            y_cut = y_to_cut[match_xdelta_condition]
            x_cut = broader_x[match_xdelta_condition]

        case 'shift_to_match_y':

            min_y = min(y_to_maintain)

            #yi = 
            
            #y_dif = y_to_cut - min_y
            #y_dif == 0

            if np.all(y_to_cut < min_y):

                print('All small')
                return np.nan, np.nan, np.nan

            xi = broader_x[y_to_cut >= min_y][0]

            time_cut_condition = broader_x >= xi

            broader_x_cut = broader_x[time_cut_condition] - xi
            match_xdelta_condition = np.isin(broader_x_cut.round(3), (narrower_x - min(narrower_x)).round(3))

            y_cut = y_to_cut[time_cut_condition][match_xdelta_condition]
            x_cut = broader_x_cut[match_xdelta_condition] + xi


    if len(y_cut) != len(y_to_maintain):

        print(xi)
        print(len(y_cut) - len(y_to_maintain))        
        
        return np.nan, np.nan, np.nan

    
    rmsd = rmsd_func(y_cut, y_to_maintain)

    return rmsd, y_cut, x_cut


def rmsd_func_many(y1, y2, x1 = False, x2 = False, mode = 'simple'):

    error = []
    cut_sim = []
    cut_std = []
    cut_t = []

    for i in range(len(y1)):
        e, y_c, t_c = rmsd_func_advance(y1[i], y2, x1 = x1, x2 = x2, mode = mode)
        
        error.append(e)
        cut_sim.append(y_c)
        cut_t.append(t_c)
        #cut_std.append(sims_std[2][i][np.isin(tt, t_c)])

    return error, cut_sim, cut_t
        

def save_sim(sim, name, file_format):

    match file_format:
        case 'npz':
            np.savez_compressed(name,
                    parameters = sim[0], susceptible = sim[1],
                    infected = sim[2], removed = sim[3])

        case 'csv':
            df = pd.DataFrame({'beta': sim[0][:,0], 'gamma': sim[0][:,1], 'n_sample': sim[0][:,2],
                               'suceptible': sim[1].tolist(), 'infected': sim[2].tolist(),
                               'removed': sim[3].tolist()})
            df.to_csv('{}.csv'.format(name), index=False)



# Funções com as taxas de variações temporal dos grupos S, I e R
def dS_dt(S, I, beta):
  return -beta*S*I

def dI_dt(S, I, beta, gamma):
  return beta*S*I - gamma*I

def dR_dt(I, gamma):
  return gamma*I

# Funções para simular epidemias via modelo SIR
# A função normaliza os valores de S, I e R
def SIR_edo(S_i, I_i, R_i, beta, gamma, dt, tf):

  t_i = 0
  tt = np.arange(t_i, tf + dt/2, dt)

  N = S_i + I_i + R_i

  S_i = S_i/N
  I_i = I_i/N
  R_i = R_i/N

  S = [S_i]
  I = [I_i]
  R = [R_i]

  for j in range(len(tt)-1):

    
    S.append(S[j] + dS_dt(S[j], I[j], beta)*dt)
    I.append(I[j] + dI_dt(S[j], I[j], beta, gamma)*dt)
    R.append(R[j] + dR_dt(I[j], gamma)*dt)

  return tt, np.array(S)*N, np.array(I)*N, np.array(R)*N


def ajuste_SIR_edo(tempo_ajuste, beta, gamma, system = 'boarding_school', acumulado = False):

    match system:

        case 'boarding_school':           
            tempo, suscetiveis, infectados, removidos = SIR_edo(762, 1, 0, beta, gamma, dt=0.1, tf=15)

        case 'diamond_princess':           
            tempo, suscetiveis, infectados, removidos = SIR_edo(3700, 1, 0, beta, gamma, dt=0.1, tf=50)

    if acumulado:

        interp_input = infectados + removidos

    else:

        interp_input = infectados

    I_interpolado = np.interp(tempo_ajuste, tempo, interp_input)
    
    return I_interpolado

## Boarding School

### Importando dados

Valores reais extraidos deste [artigo](https://www.researchgate.net/publication/339493172_On_Parameter_Estimation_Approaches_for_Predicting_Disease_Transmission_Through_Optimization_Deep_Learning_and_Statistical_Inference_Methods).

In [ ]:
sim = np.load('sir_agents_dt01_21ago2026_boardingschool.npz')

In [ ]:
boarding_school = pd.read_csv('boarding_school_data.csv')

In [ ]:
boarding_school = pd.DataFrame({'Day': np.arange(1,15),
                                'Infected number': [3, 8, 28, 75, 221, 291, 255, 235, 190, 126, 70, 28, 12, 5]})

In [ ]:
#boarding_school['Day'] = boarding_school['Day'] - 1
#boarding_school

### Simular com agentes

In [ ]:
# Condições iniciais
N, S_i = 763, 762
I_i = N - S_i
R_i = 0
init_conditions = np.array([S_i, I_i, R_i])

# Parâmetros epidêmicos
#beta, gamma = np.arange(1.5, 2.5, 0.01), np.arange(0.01, 1, 0.01)
#beta, gamma = np.array([1.66]), np.arange(0.01, 1, 0.001)
beta, gamma = np.arange(1.5, 2.5, 0.001), np.array([0.440])

dt, tf = 0.1, 20

sample_size = 100

params = np.array([[b, g, n] 
                   for b in beta for g in gamma for n in range(sample_size)],
                    dtype = np.float64)

params = sim['parameters']

# Rodar simulações
sim = abm.many_sims_numba('SIR_delay_numba', init_conditions, params, dt, tf)

# Salvar simulações
save_sim(sim=sim, name='sir_agents_dt01_01set2026_boradingschool_delay', file_format='npz')

In [ ]:
popt, pcov = curve_fit(ajuste_SIR_edo, boarding_school['Day'], boarding_school['Infected number'], p0=[1, 1], bounds=(0, [10, 10]))
pstd = np.sqrt(pcov)
boarding_school_opt_edo = SIR_edo(S_i, I_i, R_i, popt[0], popt[1], dt, tf)

### Definir valores para acessar dados

In [ ]:
parameters, susceptible, infected, removed = 0, 1, 2, 3
#parameters, susceptible, infected, removed = 'parameters', 'susceptible', 'infected', 'removed'

### Calcular RMSD

In [ ]:
tt = (np.arange(len(sim[infected][0]))*dt).round(1)

error, cut_sim, cut_t = rmsd_func_many(sim[infected], boarding_school['Infected number'].values, 
                                x1 = tt, x2 = boarding_school['Day'].values, 
                                mode = 'shift_to_match_y')

### Gráficos

#### 100 simulações de menor RMSD

In [ ]:
t_real_data = boarding_school['Day']
ind = np.argsort(error)
n_best = 1000

best_par = sim[parameters][ind[:n_best]]
par_means = np.mean(best_par, axis=0)
par_stds = np.std(best_par, axis=0) * 2
best_ti = [t_real_data[0] - cut_t[ind[i]][0] for i in range(n_best)]
ti_mean = np.mean(best_ti)
ti_std = np.std(best_ti) * 2

# Calcular infectados com EDOS usando média dos parâmetros
#S_i, I_i, R_i = sim['susceptible'][0,0], sim['infected'][0,0], sim['removed'][0,0]
S_i, I_i, R_i = 762, 1, 0
beta, gamma, dt, tf = par_means[0], par_means[1], 0.1, 20
sim_edo = SIR_edo(S_i, I_i, R_i, beta, gamma, dt, tf)

fig, ax = plt.subplots()

case_title = 'Boarding School'
sample_description = '{} simulações de menor RMSD (MBA com delay)'.format(n_best)
initial_conditions_label = r'$S(t_{{i}}) = {}, I(t_{{i}}) = {}, R(t_{{i}}) = {}$'.format(S_i, I_i, R_i)
parameters_label = r'$t_{{i}} = {} \pm {}, \beta = {} \pm {}, \gamma = {} \pm {}$'.format(ti_mean.round(1), ti_std.round(1), 
                                                                                            par_means[0].round(1), par_stds[0].round(1), 
                                                                                            par_means[1].round(2), par_stds[1].round(2))
ax.set_title(case_title+'\n'+sample_description+'\n'+initial_conditions_label+', '+parameters_label)

best_I = sim[infected][ind[0:n_best]]

for i in range(n_best):

    ti = t_real_data[0] - cut_t[ind[i]][0]
    ax.plot(tt + ti, best_I[i])

ax.plot([],[], '-', c='tab:blue', label = 'SIR (MBA)')
ax.plot(sim_edo[0] + ti_mean, sim_edo[2], '-', c = 'k', label = 'SIR (EDO) par medios')
ax.plot(boarding_school_opt_edo[0], boarding_school_opt_edo[2], '--', c = 'gold', label = 'SIR (EDO) par opt\n'
                                                                                            +r'$\beta = {:.2f} \pm {:.2f}$'.format(popt[0], pstd[0,0])+'\n'
                                                                                            +r'$\gamma = {:.2f} \pm {:.2f}$'.format(popt[1], pstd[1,1]))
ax.plot(t_real_data, boarding_school['Infected number'], 'o', c='k', label='Dados reais')

ax.set(xlabel='Tempo, t', ylabel='Infectados, I')
ax.grid()
ax.legend()

#fig.savefig('boarding_school_1000bestsims_delay.jpg', dpi = 300, format = 'jpg', bbox_inches="tight")

#### Menor RMSD com $\beta$ e $\gamma$ fixos (um de cada vez)

Fixar parametros nos valores ajustados pelo Murray, $\beta = 1.66$ e $\gamma = 0.440$

In [ ]:
t_real_data = boarding_school['Day']
ind = np.argsort(error)
n_best = 100

select_condition = {'beta': sim[parameters][:,0].round(2) == 1.66, 
                    'gamma':sim[parameters][:,1].round(2) == 0.44}

fig, ax = plt.subplots(1,2, figsize=(10,5), sharey=True, layout='tight')



case_title = 'Boarding School'
sample_description = '{} simulações de menor RMSD (MBA com delay)'.format(n_best)
fig.suptitle(case_title+'\n'+sample_description, y=0.95)

for j in range(2):

    p = list(select_condition.keys())[j]

    ind_fix_p = np.where(select_condition[p])[0]
    ind_best_fix_p = ind[np.isin(ind, ind_fix_p)]
    
    best_par = sim[parameters][ind_best_fix_p[:n_best]]
    par_means = np.mean(best_par, axis=0)
    par_stds = np.std(best_par, axis=0) * 2
    best_ti = [t_real_data[0] - cut_t[ind_best_fix_p[i]][0] for i in range(n_best)]
    ti_mean = np.mean(best_ti)
    
    ti_std = np.std(best_ti) * 2

    # Calcular infectados com EDOS usando média dos parâmetros
    #S_i, I_i, R_i = sim['susceptible'][0,0], sim['infected'][0,0], sim['removed'][0,0]
    S_i, I_i, R_i = 762, 1, 0
    beta, gamma, dt, tf = par_means[0], par_means[1], 0.1, 20
    sim_edo = SIR_edo(S_i, I_i, R_i, beta, gamma, dt, tf)

    sample_description = '{} fixo'.format([r'$\beta$', r'$\gamma$'][j])
    initial_conditions_label = r'$S(t_{{i}}) = {}, I(t_{{i}}) = {}, R(t_{{i}}) = {}$'.format(S_i, I_i, R_i)
    parameters_label = r'$t_{{i}} = {} \pm {}, \beta = {} \pm {}, \gamma = {} \pm {}$'.format(ti_mean.round(1), ti_std.round(1), 
                                                                                                par_means[0].round(2), par_stds[0].round(2), 
                                                                                                par_means[1].round(2), par_stds[1].round(2))
    ax[j].set_title(sample_description+'\n'+initial_conditions_label+'\n'+parameters_label)

    best_I = sim[infected][ind_best_fix_p[0:n_best]]

    for i in range(n_best):

        ti = t_real_data[0] - cut_t[ind_best_fix_p[i]][0]
        ax[j].plot(tt + ti, best_I[i])

    ax[j].plot([],[], '-', c='tab:blue', label = 'SIR (MBA)')
    ax[j].plot(sim_edo[0] + ti_mean, sim_edo[2], c = 'k', label = 'SIR (EDO) par medios')
    ax[j].plot(boarding_school_opt_edo[0], boarding_school_opt_edo[2], '--', c = 'gold', label = 'SIR (EDO) par opt\n'
                                                                                            +r'$\beta = {:.2f} \pm {:.2f}$'.format(popt[0], pstd[0,0])+'\n'
                                                                                            +r'$\gamma = {:.2f} \pm {:.2f}$'.format(popt[1], pstd[1,1]))

for a in ax:
    a.plot(t_real_data, boarding_school['Infected number'], 'o', c='k', label='Dados reais')
    a.set(xlabel='Tempo, t', ylabel='Infectados, I')
    a.label_outer()
    a.grid()

ax[1].legend()

#fig.savefig('boarding_school_100bestsims_fixedpar_delay.jpg', dpi = 300, format = 'jpg', bbox_inches="tight")

#### Média de menor RMSD

In [ ]:
ext_limit = 50 # Indice do valor de infectados para verificação de extinção
ext = np.apply_along_axis(identify_ext, 1, sim[infected][:,:ext_limit])
ext_reshaped = ext.reshape(int(len(ext)/sample_size), sample_size)
ext_sum = np.sum(ext_reshaped, axis = 1)

new_shape = int(len(sim[parameters])/sample_size), sample_size, int(tf/dt + 1)

S_reshaped = sim[susceptible].reshape(new_shape)
I_reshaped = sim[infected].reshape(new_shape)
R_reshaped = sim[removed].reshape(new_shape)

sims_mean = [sim[parameters][::sample_size], [], [], []]
sims_std = [sim[parameters][::sample_size], [], [], []]

for i in range(int(len(ext)/sample_size)):

    ext_condition = ext_reshaped[i] == 0

    S_mean = np.mean(S_reshaped[i][ext_condition], axis=0)
    I_mean = np.mean(I_reshaped[i][ext_condition], axis=0)
    R_mean = np.mean(R_reshaped[i][ext_condition], axis=0)

    S_std = np.std(S_reshaped[i][ext_condition], axis=0)
    I_std = np.std(I_reshaped[i][ext_condition], axis=0)
    R_std = np.std(R_reshaped[i][ext_condition], axis=0)

    sims_mean[1].append(S_mean); sims_mean[2].append(I_mean); sims_mean[3].append(R_mean)
    sims_std[1].append(S_std); sims_std[2].append(I_std); sims_std[3].append(R_std)

#sims_mean = sim['parameters'][::sample_size], np.mean(S_reshaped, axis=1), np.mean(I_reshaped, axis=1), np.mean(R_reshaped, axis=1)
#sims_std = sim['parameters'][::sample_size], np.std(S_reshaped, axis=1), np.std(I_reshaped, axis=1), np.std(R_reshaped, axis=1)


In [ ]:
tt = (np.arange(len(sim[infected][0]))*dt).round(1)

error, cut_sim, cut_t = rmsd_func_many(sims_mean[2], boarding_school['Infected number'].values, 
                                x1 = tt, x2 = boarding_school['Day'].values, 
                                mode = 'shift_to_match_y')

In [ ]:
t_real_data = boarding_school['Day']
ind = np.argsort(error)
n_best = 1

main_sum = sample_size - ext_sum[ind][0]

print(np.mean(sims_mean[0][ind][:100], axis=0))
print(np.std(sims_mean[0][ind][:100], axis=0) * 2)

best_par = sims_mean[0][ind[:n_best]]
par_means = np.mean(best_par, axis=0)
par_stds = np.std(best_par, axis=0) * 2
best_ti = [t_real_data[0] - cut_t[ind[i]][0] for i in range(n_best)]
ti_mean = np.mean(best_ti)
ti_std = np.std(best_ti) * 2

# Calcular infectados com EDOS usando média dos parâmetros
#S_i, I_i, R_i = sim['susceptible'][0,0], sim['infected'][0,0], sim['removed'][0,0]
S_i, I_i, R_i = 762, 1, 0
beta, gamma, dt, tf = par_means[0], par_means[1], 0.1, 20
sim_edo = SIR_edo(S_i, I_i, R_i, beta, gamma, dt, tf)

fig, ax = plt.subplots()

case_title = 'Boarding School'
sample_description = '{} simulações (excluindo {} extinções) da média de menor RMSD\n(MBA com delay)'.format(main_sum, ext_sum[ind][0])
initial_conditions_label = r'$S(t_{{i}}) = {}, I(t_{{i}}) = {}, R(t_{{i}}) = {}$'.format(S_i, I_i, R_i)
parameters_label = r'$t_{{i}} = {}, \beta = {}, \gamma = {}$'.format(ti_mean.round(1), par_means[0].round(1), par_means[1].round(2))
main_sum = sample_size - ext_sum[ind][0]
ax.set_title(case_title+'\n'+sample_description+'\n'+initial_conditions_label+', '+parameters_label)

ti = t_real_data[0] - cut_t[ind[0]][0]

for i in range(main_sum):

    #ax.plot(boarding_school['Day'], cut_sim[ind[i]], label='SIR (MBA)')
    #ax.plot(np.arange(len(sims_mean[2][ind[i]]))*dt, sims_mean[2][ind[i]], label='SIR (MBA)')
    #mean = cut_sim[ind[i]]
    #std = cut_std[ind[i]]
    #ax.fill_between(boarding_school['Day'], mean-std, mean+std, alpha=0.2)
    
    ax.plot(tt + ti, I_reshaped[ind[0]][ext_reshaped[ind[0]] == 0][i])


ax.plot([],[], '-', c = 'tab:blue', label='SIR (MBA) {} sims'.format(main_sum))
ax.plot(tt + ti, sims_mean[2][ind[0]], ':', c = 'k', label = 'SIR (MBA) média')
ax.plot(sim_edo[0] + ti, sim_edo[2], c = 'k', label = 'SIR (EDO) par medios')
ax.plot(boarding_school_opt_edo[0], boarding_school_opt_edo[2], '--', c = 'gold', label = 'SIR (EDO) par opt\n'
                                                                                            +r'$\beta = {:.2f} \pm {:.2f}$'.format(popt[0], pstd[0,0])+'\n'
                                                                                            +r'$\gamma = {:.2f} \pm {:.2f}$'.format(popt[1], pstd[1,1]))
ax.plot(t_real_data, boarding_school['Infected number'], 'o', c='k', label='Dados reais')


ax.set(xlabel='Tempo, t', ylabel='Infectados, I')
ax.grid()
ax.legend()

#fig.savefig('boarding_school_allsims_bestmean_delay.jpg', dpi = 300, format = 'jpg', bbox_inches="tight")

## Diamond Princess

De acordo com este [artigo](https://pmc.ncbi.nlm.nih.gov/articles/PMC7107563/) o número de pessoas no Diamond Princess era 3700.

Os dados de valores casos e mortes confirmadas foram extraidos dos [relatórios da OMS](https://www.who.int/emergencies/diseases/novel-coronavirus-2019/situation-reports/situation-reports-archive). Perceba que existem quedas no número de casos acumulados, isso se deve a correções realizadas pela OMS.

### Importar dados

In [ ]:
sim = np.load('sir_agents_dt01_01set2026_diamondprincess.npz')

In [ ]:
sim

In [ ]:
diamond_princess = pd.read_csv('diamond_princess_data.csv')

In [ ]:
#diamond_princess = diamond_princess[:30]

### Simular com agentes

In [ ]:
# Condições iniciais
N, S_i = 3700, 3699
I_i = N - S_i
R_i = 0
init_conditions = np.array([S_i, I_i, R_i])

# Parâmetros epidêmicos
beta, gamma = np.arange(0.5, 4, 0.05), np.arange(0.5, 4, 0.05)

dt, tf = 0.1, 50

sample_size = 100

params = np.array([[b, g, n] 
                   for b in beta for g in gamma for n in range(sample_size)],
                    dtype = np.float64)


# Rodar simulações
#sim = abm.many_sims_numba('SIR_delay_numba', init_conditions, params, dt, tf)

# Salvar simulações
#save_sim(sim=sim, name='sir_agents_dt01_01set2026_diamondprincess_delay', file_format='npz')

In [ ]:
wrapped_func = ft.partial(ajuste_SIR_edo, system = 'diamond_princess', acumulado=True)
popt, pcov = curve_fit(wrapped_func, diamond_princess['time_delta'], diamond_princess['confirmed_cases'], p0=[2, 2], bounds=(1, [10, 10]))
pstd = np.sqrt(pcov)
diamond_princess_opt_edo = SIR_edo(S_i, I_i, R_i, popt[0], popt[1], dt, tf)

### Definir valores para acessar dados

In [ ]:
#parameters, susceptible, infected, removed = 0, 1, 2, 3
parameters, susceptible, infected, removed = 'parameters', 'susceptible', 'infected', 'removed'

### Remover extinções e calcular acumulados

In [ ]:
ext_limit = 100 # Indice do valor de infectados para verificação de extinção
ext = np.apply_along_axis(identify_ext, 1, sim[infected][:,:ext_limit])

In [ ]:
sims_cut_ext = [sim[parameters][ext == 0], 
                sim[susceptible][ext == 0], 
                sim[infected][ext == 0], 
                sim[removed][ext == 0]]
#sims_std = [sim['parameters'][ext == 0], [], [], []]

In [ ]:
sum(ext)/len(ext)

Porcentagem de extinções com delay: 88.0%; sem delay: 79.0%

In [ ]:
len(ext) - sum(ext)

In [ ]:
C_I = N-np.array(sim[susceptible])
#I_cum_std = np.array(sims_std[1])

### Calcular RMSD

In [ ]:
tt = (np.arange(len(C_I[0]))*dt).round(1)

error, cut_sim, cut_t = rmsd_func_many(C_I, diamond_princess['confirmed_cases'].values, 
                                    x1 = tt, x2 = diamond_princess['time_delta'].values, 
                                    mode = 'shift_to_match_y')


### Gráficos

#### 100 simulações de menor RMSD

In [ ]:
t_real_data = diamond_princess['time_delta']
ind = np.argsort(error)
n_best = 1000

best_par = sim[parameters][ind[:n_best]]
par_means = np.mean(best_par, axis=0)
par_stds = np.std(best_par, axis=0) * 2
best_ti = [t_real_data[0] - cut_t[ind[i]][0] for i in range(n_best)]
ti_mean = np.mean(best_ti)
ti_std = np.std(best_ti) * 2

# Calcular infectados com EDOS usando média dos parâmetros
#S_i, I_i, R_i = sim['susceptible'][0,0], sim['infected'][0,0], sim['removed'][0,0]
S_i, I_i, R_i = 3700, 1, 0
beta, gamma, dt, tf = par_means[0], par_means[1], 0.1, 50
sim_edo = SIR_edo(S_i, I_i, R_i, beta, gamma, dt, tf)

fig, ax = plt.subplots()

case_title = 'Diamond Princess'
sample_description = '{} simulações de menor RMSD (MBA com delay)'.format(n_best)
initial_conditions_label = r'$S(t_{{i}}) = {}, I(t_{{i}}) = {}, R(t_{{i}}) = {}$'.format(S_i, I_i, R_i)
parameters_label = r'$t_{{i}} = {} \pm {}, \beta = {} \pm {}, \gamma = {} \pm {}$'.format(ti_mean.round(1), ti_std.round(1), 
                                                                                            par_means[0].round(1), par_stds[0].round(1), 
                                                                                            par_means[1].round(2), par_stds[1].round(2))
ax.set_title(case_title+'\n'+sample_description+'\n'+initial_conditions_label+'\n'+parameters_label)

best_C_I = C_I[ind[0:n_best]]

for i in range(n_best):

    ti = t_real_data[0] - cut_t[ind[i]][0]
    ax.plot(tt + ti, best_C_I[i])

ax.plot([],[], '-', c='tab:blue', label = 'SIR (MBA)')
ax.plot(sim_edo[0] + ti_mean, N-sim_edo[1], '-', c = 'k', label = 'SIR (EDO) par medios')

ax.plot(diamond_princess_opt_edo[0], N-diamond_princess_opt_edo[1], '--', c = 'gold', label = 'SIR (EDO) par opt\n'
                                                                                            +r'$\beta = {:.2f} \pm {:.2f}$'.format(popt[0], pstd[0,0])+'\n'
                                                                                            +r'$\gamma = {:.2f} \pm {:.2f}$'.format(popt[1], pstd[1,1]))

ax.plot(t_real_data, diamond_princess['confirmed_cases'], 'o', c='k', label='Dados reais')

ax.set(xlim=(-5, 50), xlabel='Tempo, t', ylabel='Acumulado de infectados, $C_{I}$')
ax.grid()
ax.legend()

#fig.savefig('diamond_princess_1000bestsims_delay.jpg', dpi = 300, format = 'jpg', bbox_inches="tight")

#### Menor RMSD com $\beta$ e $\gamma$ fixos (um de cada vez)

Fixar parametros nos valores ajustados por ????, $\beta = ????$ e $\gamma = ????$

In [ ]:
t_real_data = diamond_princess['time_delta']
ind = np.argsort(error)
n_best = 100

select_condition = {'beta': sim[parameters][:,0].round(2) == 1.66, 
                    'gamma':sim[parameters][:,1].round(2) == 0.44}

fig, ax = plt.subplots(1,2, figsize=(10,5), sharey=True, layout='tight')



case_title = 'Boarding School'
sample_description = '{} simulações de menor RMSD (MBA com delay)'.format(n_best)
fig.suptitle(case_title+'\n'+sample_description, y=0.95)

for j in range(2):

    p = list(select_condition.keys())[j]

    ind_fix_p = np.where(select_condition[p])[0]
    ind_best_fix_p = ind[np.isin(ind, ind_fix_p)]
    
    best_par = sim[parameters][ind_best_fix_p[:n_best]]
    par_means = np.mean(best_par, axis=0)
    par_stds = np.std(best_par, axis=0) * 2
    best_ti = [t_real_data[0] - cut_t[ind_best_fix_p[i]][0] for i in range(n_best)]
    ti_mean = np.mean(best_ti)
    
    ti_std = np.std(best_ti) * 2

    # Calcular infectados com EDOS usando média dos parâmetros
    #S_i, I_i, R_i = sim['susceptible'][0,0], sim['infected'][0,0], sim['removed'][0,0]
    S_i, I_i, R_i = , 1, 0
    beta, gamma, dt, tf = par_means[0], par_means[1], 0.1, 20
    sim_edo = SIR_edo(S_i, I_i, R_i, beta, gamma, dt, tf)

    sample_description = '{} fixo'.format([r'$\beta$', r'$\gamma$'][j])
    initial_conditions_label = r'$S(t_{{i}}) = {}, I(t_{{i}}) = {}, R(t_{{i}}) = {}$'.format(S_i, I_i, R_i)
    parameters_label = r'$t_{{i}} = {} \pm {}, \beta = {} \pm {}, \gamma = {} \pm {}$'.format(ti_mean.round(1), ti_std.round(1), 
                                                                                                par_means[0].round(2), par_stds[0].round(2), 
                                                                                                par_means[1].round(2), par_stds[1].round(2))
    ax[j].set_title(sample_description+'\n'+initial_conditions_label+'\n'+parameters_label)

    best_I = sim[infected][ind_best_fix_p[0:n_best]]

    for i in range(n_best):

        ti = t_real_data[0] - cut_t[ind_best_fix_p[i]][0]
        ax[j].plot(tt + ti, best_I[i])

    ax[j].plot([],[], '-', c='tab:blue', label = 'SIR (MBA)')
    ax[j].plot(sim_edo[0] + ti_mean, sim_edo[2], c = 'k', label = 'SIR (EDO) par medios')
    ax[j].plot(boarding_school_opt_edo[0], boarding_school_opt_edo[2], '--', c = 'gold', label = 'SIR (EDO) par opt\n'
                                                                                            +r'$\beta = {:.2f} \pm {:.2f}$'.format(popt[0], pstd[0,0])+'\n'
                                                                                            +r'$\gamma = {:.2f} \pm {:.2f}$'.format(popt[1], pstd[1,1]))

for a in ax:
    a.plot(t_real_data, boarding_school['Infected number'], 'o', c='k', label='Dados reais')
    a.set(xlabel='Tempo, t', ylabel='Infectados, I')
    a.label_outer()
    a.grid()

ax[1].legend()

#fig.savefig('boarding_school_100bestsims_fixedpar_delay.jpg', dpi = 300, format = 'jpg', bbox_inches="tight")

In [ ]:
ind = np.argsort(error)
n_mean = 100

best_par = sims_mean[0][ind][:n_mean]
par_means = np.mean(best_par, axis=0)
par_stds = np.std(best_par, axis=0) *2
best_ti = [cut_t[ind[i]][0] for i in range(n_mean)]
ti_mean = np.mean(best_ti)
ti_std = np.std(best_ti) * 2

fig, ax = plt.subplots()

initial_conditions_label = r'$S(t_{{i}}) = {}, I(t_{{i}}) = {}, R(t_{{i}}) = {}$'.format(S_i, I_i, R_i)
parameters_label = r'$t_{{i}} = {} \pm {}, \beta = {} \pm {}, \gamma = {} \pm {}$'.format(-1*ti_mean.round(1), ti_std.round(1), par_means[0].round(1), par_stds[0].round(1), par_means[1].round(1), par_stds[1].round(1))
ax.set_title('Diamond Princess'+'\n'+'Média de {} simulações'.format(n_mean)+'\n'+initial_conditions_label+'\n'+parameters_label)

for i in range(n_mean):

    #ax.plot(diamond_princess['time_delta'], cut_sim[ind[i]], 'o-', label='SIR (MBA)')
    #mean = cut_sim[ind[i]]
    #std = cut_std[ind[i]]
    #ax.fill_between(diamond_princess['time_delta'], mean-std, mean+std, alpha=0.2, label='1 d. padrão')

    
    ti = cut_t[ind[i]][0]
    ax.plot(tt - ti, I_cum_mean[ind[i]])
    #mean = I_cum_mean[ind[i]]
    #std = I_cum_std[ind[i]]
    #ax.fill_between(tt - ti, mean - std, mean + std, alpha=0.2, label='1 d. padrão')
    #ax.plot([],[], ' ', label = r'$t_{{i}} = {}$'.format(-1*ti), c='y')
    
ax.plot([],[], '-', label = 'SIR (MBA)', c='tab:blue')

ax.plot(diamond_princess['time_delta'], diamond_princess['confirmed_cases'], 'o--', c='k', label='Dados reais')

ax.set(xlim=(-2, 43), xlabel='Tempo, t', ylabel='Acumulado de infectados, $C_{I}$')
ax.grid()
ax.legend()

#fig.savefig('diamond_princess_media.jpg', dpi = 300, format = 'jpg', bbox_inches="tight")

#### Média de menor RMSD

In [ ]:
ext_limit = 100 # Indice do valor de infectados para verificação de extinção
ext = np.apply_along_axis(identify_ext, 1, sim[infected][:,:ext_limit])
ext_reshaped = ext.reshape(int(len(ext)/sample_size), sample_size)
ext_sum = np.sum(ext_reshaped, axis = 1)

new_shape = int(len(sim[parameters])/sample_size), sample_size, int(tf/dt + 1)

S_reshaped = sim[susceptible].reshape(new_shape)
I_reshaped = sim[infected].reshape(new_shape)
R_reshaped = sim[removed].reshape(new_shape)

sims_mean = [sim[parameters][::sample_size], [], [], []]
sims_std = [sim[parameters][::sample_size], [], [], []]

for i in range(int(len(ext)/sample_size)):

    ext_condition = ext_reshaped[i] == 0

    S_mean = np.mean(S_reshaped[i][ext_condition], axis=0)
    I_mean = np.mean(I_reshaped[i][ext_condition], axis=0)
    R_mean = np.mean(R_reshaped[i][ext_condition], axis=0)

    S_std = np.std(S_reshaped[i][ext_condition], axis=0)
    I_std = np.std(I_reshaped[i][ext_condition], axis=0)
    R_std = np.std(R_reshaped[i][ext_condition], axis=0)

    sims_mean[1].append(S_mean); sims_mean[2].append(I_mean); sims_mean[3].append(R_mean)
    sims_std[1].append(S_std); sims_std[2].append(I_std); sims_std[3].append(R_std)

#sims_mean = sim['parameters'][::sample_size], np.mean(S_reshaped, axis=1), np.mean(I_reshaped, axis=1), np.mean(R_reshaped, axis=1)
#sims_std = sim['parameters'][::sample_size], np.std(S_reshaped, axis=1), np.std(I_reshaped, axis=1), np.std(R_reshaped, axis=1)


In [ ]:
sum(ext)/len(ext)

In [ ]:
C_I = N-np.array(sims_mean[1])
#I_cum_std = np.array(sims_std[1])

In [ ]:
tt = (np.arange(len(C_I[0]))*dt).round(1)

error, cut_sim, cut_t = rmsd_func_many(C_I, diamond_princess['confirmed_cases'].values, 
                                    x1 = tt, x2 = diamond_princess['time_delta'].values, 
                                    mode = 'shift_to_match_y')

In [ ]:
t_real_data = diamond_princess['time_delta']
ind = np.argsort(error)
n_best = 1

main_sum = sample_size - ext_sum[ind][0]

print(np.mean(sims_mean[0][ind][:100], axis=0))
print(np.std(sims_mean[0][ind][:100], axis=0) * 2)

best_par = sims_mean[0][ind[:n_best]]
par_means = np.mean(best_par, axis=0)
par_stds = np.std(best_par, axis=0) * 2
best_ti = [t_real_data[0] - cut_t[ind[i]][0] for i in range(n_best)]
ti_mean = np.mean(best_ti)
ti_std = np.std(best_ti) * 2

# Calcular infectados com EDOS usando média dos parâmetros
#S_i, I_i, R_i = sim['susceptible'][0,0], sim['infected'][0,0], sim['removed'][0,0]
S_i, I_i, R_i = 3700, 1, 0
beta, gamma, dt, tf = par_means[0], par_means[1], 0.1, 50
sim_edo = SIR_edo(S_i, I_i, R_i, beta, gamma, dt, tf)

fig, ax = plt.subplots()

case_title = 'Diamond Princess'
sample_description = '{} simulações (excluindo {} extinções) da média de menor RMSD\n(MBA com delay)'.format(main_sum, ext_sum[ind][0])
initial_conditions_label = r'$S(t_{{i}}) = {}, I(t_{{i}}) = {}, R(t_{{i}}) = {}$'.format(S_i, I_i, R_i)
parameters_label = r'$t_{{i}} = {}, \beta = {}, \gamma = {}$'.format(ti_mean.round(1), par_means[0].round(1), par_means[1].round(2))
main_sum = sample_size - ext_sum[ind][0]
ax.set_title(case_title+'\n'+sample_description+'\n'+initial_conditions_label+', '+parameters_label)

ti = t_real_data[0] - cut_t[ind[0]][0]

for i in range(main_sum):
    
    ax.plot(tt + ti, N - S_reshaped[ind[0]][ext_reshaped[ind[0]] == 0][i])

ax.plot([],[], '-', c = 'tab:blue', label='SIR (MBA) {} sims'.format(main_sum))
ax.plot(tt + ti, C_I[ind[0]], ':', c = 'k', label = 'SIR (MBA) média')
ax.plot(sim_edo[0] + ti, N-sim_edo[1], c = 'k', label = 'SIR (EDO) par medios')
ax.plot(diamond_princess_opt_edo[0], N-diamond_princess_opt_edo[1], '--', c = 'gold', label = 'SIR (EDO) par opt\n'
                                                                                            +r'$\beta = {:.2f} \pm {:.2f}$'.format(popt[0], pstd[0,0])+'\n'
                                                                                            +r'$\gamma = {:.2f} \pm {:.2f}$'.format(popt[1], pstd[1,1]))
ax.plot(t_real_data, diamond_princess['confirmed_cases'], 'o', c='k', label='Dados reais')


ax.set(xlim=(-5, 50), xlabel='Tempo, t', ylabel='Acumulado de infectados, $C_{I}$')
ax.grid()
ax.legend(loc='lower right')

#fig.savefig('diamond_princess_allsims_bestmean_delay.jpg', dpi = 300, format = 'jpg', bbox_inches="tight")

### Gráficos

#### 100 simulações de menor RMSD

In [ ]:
t_real_data = boarding_school['Day']
ind = np.argsort(error)
n_best = 1000

best_par = sim[parameters][ind[:n_best]]
par_means = np.mean(best_par, axis=0)
par_stds = np.std(best_par, axis=0) * 2
best_ti = [t_real_data[0] - cut_t[ind[i]][0] for i in range(n_best)]
ti_mean = np.mean(best_ti)
ti_std = np.std(best_ti) * 2

# Calcular infectados com EDOS usando média dos parâmetros
#S_i, I_i, R_i = sim['susceptible'][0,0], sim['infected'][0,0], sim['removed'][0,0]
S_i, I_i, R_i = 762, 1, 0
beta, gamma, dt, tf = par_means[0], par_means[1], 0.1, 20
sim_edo = SIR_edo(S_i, I_i, R_i, beta, gamma, dt, tf)

fig, ax = plt.subplots()

case_title = 'Boarding School'
sample_description = '{} simulações de menor RMSD (MBA com delay)'.format(n_best)
initial_conditions_label = r'$S(t_{{i}}) = {}, I(t_{{i}}) = {}, R(t_{{i}}) = {}$'.format(S_i, I_i, R_i)
parameters_label = r'$t_{{i}} = {} \pm {}, \beta = {} \pm {}, \gamma = {} \pm {}$'.format(ti_mean.round(1), ti_std.round(1), 
                                                                                            par_means[0].round(1), par_stds[0].round(1), 
                                                                                            par_means[1].round(2), par_stds[1].round(2))
ax.set_title(case_title+'\n'+sample_description+'\n'+initial_conditions_label+', '+parameters_label)

best_I = sim[infected][ind[0:n_best]]

for i in range(n_best):

    ti = t_real_data[0] - cut_t[ind[i]][0]
    ax.plot(tt + ti, best_I[i])

ax.plot([],[], '-', c='tab:blue', label = 'SIR (MBA)')
ax.plot(sim_edo[0] + ti_mean, sim_edo[2], '-', c = 'k', label = 'SIR (EDO) par medios')
ax.plot(boarding_school_opt_edo[0], boarding_school_opt_edo[2], '--', c = 'gold', label = 'SIR (EDO) par opt\n'
                                                                                            +r'$\beta = {:.2f} \pm {:.2f}$'.format(popt[0], pstd[0,0])+'\n'
                                                                                            +r'$\gamma = {:.2f} \pm {:.2f}$'.format(popt[1], pstd[1,1]))
ax.plot(t_real_data, boarding_school['Infected number'], 'o', c='k', label='Dados reais')

ax.set(xlabel='Tempo, t', ylabel='Infectados, I')
ax.grid()
ax.legend()

#fig.savefig('boarding_school_1000bestsims_delay.jpg', dpi = 300, format = 'jpg', bbox_inches="tight")

#### Menor RMSD com $\beta$ e $\gamma$ fixos (um de cada vez)

Fixar parametros nos valores ajustados pelo Murray, $\beta = 1.66$ e $\gamma = 0.440$

In [ ]:
t_real_data = boarding_school['Day']
ind = np.argsort(error)
n_best = 100

select_condition = {'beta': sim[parameters][:,0].round(2) == 1.66, 
                    'gamma':sim[parameters][:,1].round(2) == 0.44}

fig, ax = plt.subplots(1,2, figsize=(10,5), sharey=True, layout='tight')



case_title = 'Boarding School'
sample_description = '{} simulações de menor RMSD (MBA com delay)'.format(n_best)
fig.suptitle(case_title+'\n'+sample_description, y=0.95)

for j in range(2):

    p = list(select_condition.keys())[j]

    ind_fix_p = np.where(select_condition[p])[0]
    ind_best_fix_p = ind[np.isin(ind, ind_fix_p)]
    
    best_par = sim[parameters][ind_best_fix_p[:n_best]]
    par_means = np.mean(best_par, axis=0)
    par_stds = np.std(best_par, axis=0) * 2
    best_ti = [t_real_data[0] - cut_t[ind_best_fix_p[i]][0] for i in range(n_best)]
    ti_mean = np.mean(best_ti)
    
    ti_std = np.std(best_ti) * 2

    # Calcular infectados com EDOS usando média dos parâmetros
    #S_i, I_i, R_i = sim['susceptible'][0,0], sim['infected'][0,0], sim['removed'][0,0]
    S_i, I_i, R_i = 762, 1, 0
    beta, gamma, dt, tf = par_means[0], par_means[1], 0.1, 20
    sim_edo = SIR_edo(S_i, I_i, R_i, beta, gamma, dt, tf)

    sample_description = '{} fixo'.format([r'$\beta$', r'$\gamma$'][j])
    initial_conditions_label = r'$S(t_{{i}}) = {}, I(t_{{i}}) = {}, R(t_{{i}}) = {}$'.format(S_i, I_i, R_i)
    parameters_label = r'$t_{{i}} = {} \pm {}, \beta = {} \pm {}, \gamma = {} \pm {}$'.format(ti_mean.round(1), ti_std.round(1), 
                                                                                                par_means[0].round(2), par_stds[0].round(2), 
                                                                                                par_means[1].round(2), par_stds[1].round(2))
    ax[j].set_title(sample_description+'\n'+initial_conditions_label+'\n'+parameters_label)

    best_I = sim[infected][ind_best_fix_p[0:n_best]]

    for i in range(n_best):

        ti = t_real_data[0] - cut_t[ind_best_fix_p[i]][0]
        ax[j].plot(tt + ti, best_I[i])

    ax[j].plot([],[], '-', c='tab:blue', label = 'SIR (MBA)')
    ax[j].plot(sim_edo[0] + ti_mean, sim_edo[2], c = 'k', label = 'SIR (EDO) par medios')
    ax[j].plot(boarding_school_opt_edo[0], boarding_school_opt_edo[2], '--', c = 'gold', label = 'SIR (EDO) par opt\n'
                                                                                            +r'$\beta = {:.2f} \pm {:.2f}$'.format(popt[0], pstd[0,0])+'\n'
                                                                                            +r'$\gamma = {:.2f} \pm {:.2f}$'.format(popt[1], pstd[1,1]))

for a in ax:
    a.plot(t_real_data, boarding_school['Infected number'], 'o', c='k', label='Dados reais')
    a.set(xlabel='Tempo, t', ylabel='Infectados, I')
    a.label_outer()
    a.grid()

ax[1].legend()

#fig.savefig('boarding_school_100bestsims_fixedpar_delay.jpg', dpi = 300, format = 'jpg', bbox_inches="tight")